## Continuous Control Comparison: PPO vs SAC vs TD3 on Pendulum-v1

This notebook imports the camera-ready algorithm modules from `cam_ready/` and compares PPO, SAC, and TD3 under the same approximate environment-step budget. It also calculates computational-complexity estimates and runs per-algorithm ablations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "cam_ready":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cam_ready.utils.complexity import continuous_pendulum_complexity
from cam_ready.utils.experiments import (
    run_continuous_pendulum_comparison,
    run_ppo_pendulum_ablations,
    run_sac_pendulum_ablations,
    run_td3_pendulum_ablations,
    summarize_final_scores,
)
from cam_ready.utils.common import plot_comparison


def as_table(rows):
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


def print_score_table(results, window):
    rows = summarize_final_scores(results, window=window)
    for row in rows:
        print(
            f"{row['algorithm']}: "
            f"{row['mean_final_return']:.2f} +/- {row['std_final_return']:.2f}"
        )
    return as_table(rows)

In [ ]:
SEEDS = [0, 16, 25]
TOTAL_ENV_STEPS = 25_000
WINDOW = 10

### Computational Complexity

Notation:
- `E`: environment steps
- `U`: gradient updates
- `d`: TD3 policy-delay interval
- `B`: minibatch size
- `R`: replay-buffer capacity
- `S`: state dimension
- `A`: continuous action dimension
- `Cpi`, `Cq`, `Cv`: one forward pass through the policy, Q, and value networks

The table below gives both the asymptotic form and a concrete proxy for this notebook's hyperparameters. `total_mac_proxy` counts linear-layer multiply-accumulates and uses the standard approximation that a backward pass costs about two forward passes. Environment physics, Python overhead, sampling, distribution bookkeeping, and target-parameter copies are excluded.

In [ ]:
complexity_rows = continuous_pendulum_complexity(total_env_steps=TOTAL_ENV_STEPS)
as_table(complexity_rows)

### Main Comparison

This is the headline continuous-control comparison: three seeds, same approximate environment-step budget, and the same smoothing window for plotting.

In [ ]:
results = run_continuous_pendulum_comparison(
    seeds=SEEDS,
    total_env_steps=TOTAL_ENV_STEPS,
    progress=True,
)

In [ ]:
fig, ax = plot_comparison(
    results,
    window=WINDOW,
    title="PPO vs SAC vs TD3 on Pendulum-v1",
    ylabel="Episode return (higher is better)",
)

In [ ]:
main_summary = print_score_table(results, window=WINDOW)
main_summary

### Individual Algorithm Ablations

Each ablation keeps the environment fixed and changes one implementation or hyperparameter group at a time. These cells use one seed and a smaller step budget by default so they are practical to iterate on; set `ABLATION_SEEDS = SEEDS` and increase `ABLATION_ENV_STEPS` for report-quality runs.

PPO ablations:
- `clip_0.10`: smaller PPO clipping range.
- `lambda_0.80`: lower GAE lambda, closer to shorter-horizon TD advantages.
- `lower_action_std`: starts continuous exploration with a smaller standard deviation.

SAC ablations:
- `alpha_0.05` and `alpha_0.50`: lower or higher entropy temperature.
- `no_random_warmup`: removes random-action replay seeding.

TD3 ablations:
- `no_target_smoothing`: removes target policy smoothing noise.
- `policy_delay_1`: updates the actor every critic update.
- `act_noise_0.20`: increases behavior-policy exploration noise.

In [ ]:
ABLATION_SEEDS = [0]
ABLATION_ENV_STEPS = 10_000

In [ ]:
ppo_ablations = run_ppo_pendulum_ablations(
    seeds=ABLATION_SEEDS,
    total_env_steps=ABLATION_ENV_STEPS,
    progress=True,
)

fig, ax = plot_comparison(
    ppo_ablations,
    window=WINDOW,
    title="PPO Pendulum-v1 Ablations",
    ylabel="Episode return (higher is better)",
)

ppo_ablation_summary = print_score_table(ppo_ablations, window=WINDOW)
ppo_ablation_summary

In [ ]:
sac_ablations = run_sac_pendulum_ablations(
    seeds=ABLATION_SEEDS,
    total_env_steps=ABLATION_ENV_STEPS,
    progress=True,
)

fig, ax = plot_comparison(
    sac_ablations,
    window=WINDOW,
    title="SAC Pendulum-v1 Ablations",
    ylabel="Episode return (higher is better)",
)

sac_ablation_summary = print_score_table(sac_ablations, window=WINDOW)
sac_ablation_summary

In [ ]:
td3_ablations = run_td3_pendulum_ablations(
    seeds=ABLATION_SEEDS,
    total_env_steps=ABLATION_ENV_STEPS,
    progress=True,
)

fig, ax = plot_comparison(
    td3_ablations,
    window=WINDOW,
    title="TD3 Pendulum-v1 Ablations",
    ylabel="Episode return (higher is better)",
)

td3_ablation_summary = print_score_table(td3_ablations, window=WINDOW)
td3_ablation_summary